# Design Spaces 

## Setup

### Import Packages and Directories

In [ ]:
from pathlib import Path
import flowtangent as ft
from flowtangent.models.kernels import RBFKernel
from flowtangent.models.surrogates import GaussianProcessSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

TEST_DIR = Path("/Users/jordan/dev/EDEN/flowtangent/exp/XFOIL").parent
DATA_DIR = Path("/Users/jordan/dev/data/")

In [15]:
import string
from functools import lru_cache

@lru_cache(maxsize=2048)
def surf_lookup(name:str):

    safe_name = name.replace("-", "_").replace(".", "_")
    if safe_name and safe_name[0] in string.digits:
        safe_name = f"af_{safe_name}"

    af = getattr(ft.airfoils, safe_name)

    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(row:dict):
    af_name = row["foil_name"].item()
    row["y_af"] = surf_lookup(af_name)
    return row

### Import Dataset

In [23]:
from functools import lru_cache
import jax.numpy as jnp

XFD = ft.utils.data.XArrayDataset(DATA_DIR / "xfoil_4D.zarr")
# XFD = XFD.transform_variable(name2surf)

In [29]:
XFD[50_000]

{'bend_data': array([ 6.67808e-02,  2.02709e+00,  4.09653e-01,  5.89206e-01,
        -4.09686e-01,  3.42036e-03,  5.80504e-03,  8.12240e-03,
         5.20574e-02, -5.01720e-02,  3.98438e-05,  7.65382e-04,
                 nan,  4.95167e-01,  5.03692e-01, -4.95200e-01,
         1.71963e-01,  3.41405e-01,  8.70048e-03,  5.14794e-02,
        -5.07501e-02,  2.76186e-03,  5.36499e-02,          nan],
       dtype=float32),
 'conditions': array([-8.3698311e+00,  8.2558113e-01,  6.9007562e+04,  1.0391619e-01],
       dtype=float32),
 'foil_name': array('e226', dtype='<U50'),
 'panel_data': array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,

In [22]:
XFD[10_000_000]["polar_data"].shape

(9,)

### Setup Manifold

In [17]:
airfold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
confold = Passthrough(dim=5) # Mach, Re, Flap Angle, Flap Loc., Alpha
mfold = airfold + confold

### Setup Surrogate

In [ ]:
krnl = RBFKernel(input_dim=21)
surr = GaussianProcessSurrogate(42, input_dim=21, output_dim=1, kernel=krnl)

### Setup Parameters and Requirements

In [ ]:
y_af = Parameter(name="y_af")
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Paramter(name="alpha")

theta = (y_af, Mach, re, flap_loc, flap_angle, alpha)

CL_max = Requirement(name="CL", lower_bound=1.2)
alpha_range = Requirement(name="alpha", lower_bound=0.0, upper_bound=5.0)
R = (CL_max, alpha_range)

### Setup Design Space

In [ ]:
airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
    requirements=R
)